# 如何为 Runnable 添加默认调用参数

有时我们希望在 `Runnable` 内部的 `RunnableSequence` 中调用具有常量参数的可运行项，

这些参数不是序列中前一个可运行项的输出的一部分，也不是用户输入的一部分。

我们可以使用 **`Runnable.bind()`** 方法提前设置这些参数。

In [1]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv("apikey.env")
BASE_URL = 'https://api.deepseek.com'
API_KEY = os.getenv('DEEPSEEK-API-KEY')
deepseek_chat_model = 'deepseek-chat'
if  not API_KEY:
    raise ValueError("WARNING: NOT FOUND OPENAI_API_KEY，PLEASE CHECK .env SETING。")
else:
    print("SECESSFULLY!")
model = ChatDeepSeek(api_key=API_KEY, base_url=BASE_URL, model=deepseek_chat_model)

SECESSFULLY!


In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Write out the following equation using algebraic symbols then solve it. Use the format\n\nEQUATION:...\nSOLUTION:...\n\n",
        ),
        ("human", "{equation_statement}"),
    ]
)

runnable = (
    {"equation_statement": RunnablePassthrough()} | prompt | model | StrOutputParser()
)

print(runnable.invoke("x raised to the third plus seven equals 12"))

Let's break it down step by step.

**Step 1: Translate words into an equation**  
"x raised to the third" means \( x^3 \).  
"plus seven" means \( + 7 \).  
"equals 12" means \( = 12 \).  

So the equation is:  
\[
x^3 + 7 = 12
\]

**Step 2: Solve for \( x \)**  
Subtract 7 from both sides:  
\[
x^3 = 12 - 7
\]
\[
x^3 = 5
\]

Take the cube root of both sides:  
\[
x = \sqrt[3]{5}
\]

---

**Final answer:**

EQUATION: \( x^3 + 7 = 12 \)  
SOLUTION: \( x = \sqrt[3]{5} \)


并希望使用某些 stop 词调用模型，以便在某些提示技术中缩短输出。

In [4]:
runnable = (
    {"equation_statement": RunnablePassthrough()}
    | prompt
    | model.bind(stop="EQUATION")
    | StrOutputParser()
)

for chunck in runnable.stream("x raised to the third plus seven equals 12"):
    print(chunck, end="", flush=True)

Let's break this down step by step.

---

**Step 1: Translate words into an equation**  
"x raised to the third" means \( x^3 \).  
"plus seven" means \( + 7 \).  
"equals 12" means \( = 12 \).  

So the equation is:  
\[
x^3 + 7 = 12
\]

---

**Step 2: Solve for \( x \)**  
Subtract 7 from both sides:  
\[
x^3 = 12 - 7
\]
\[
x^3 = 5
\]

Take the cube root of both sides:  
\[
x = \sqrt[3]{5}
\]

---

**Final answer:**



In [6]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather of a location, the user should supply a location first.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    }
                },
                "required": ["location"]
            },
        }
    },
]

In [12]:
model.invoke("What's the weather in SF, NYC and LA?")

AIMessage(content='I can help you get the weather for those cities! However, I need to check them one at a time. Let me start with San Francisco, CA.', additional_kwargs={'tool_calls': [{'id': 'call_00_QxAc8mQI4ELnhIjYf9TAwaAZ', 'function': {'arguments': '{"location": "San Francisco, CA"}', 'name': 'get_weather'}, 'type': 'function', 'index': 0}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 181, 'total_tokens': 230, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 53}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_ffc7281d48_prod0820_fp8_kvcache', 'id': 'e63ca25d-1e57-4045-9174-f79d1edcbd98', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--5a7e8256-76e0-4bfa-957c-ae4f70e8d9db-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'San Francisco, CA'}, 'id': 'call_00_QxAc8

# 如何将参数从一个步骤传递到下一个步骤
在组合多个步骤的链时，有时您希望将**前一步的数据原样传递**，以便作为后续步骤的输入。

`RunnablePassthrough` 类允许您做到这一点，通常与 `RunnableParallel` 一起使用，以将数据传递到您构建的链中的后续步骤。

In [13]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough

runnable = RunnableParallel(
    passed=RunnablePassthrough(),
    modified=lambda x : x["num"] +1
)
runnable.invoke({"num":1})

{'passed': {'num': 1}, 'modified': 2}

1. passed 键被调用了 RunnablePassthrough()，因此它简单地传递了 {'num': 1}。

2. modified这使用了lambda x : x["num"] +1, 结果是 modified 键的值为 2。

In [14]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="moka-ai/m3e-base")
text = """
“秦始皇在统一六国后，推行了‘书同文，车同轨’等一系列强化中央集权的政策。
他下令废除六国原有的多样化的度量衡标准，在全国范围内推行统一的计量单位，
方便了税收、贸易和工程建设。同时，他命丞相李斯等人创立小篆作为官方标准字体，
淘汰了六国形态各异的文字。这些措施极大地促进了帝国不同地区之间的经济文化交流和政治管理。
"""
vectorstore = FAISS.from_texts(
    [text], embedding=embedding
    )
retriever = vectorstore.as_retriever()
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
model = model

retrieval_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

for chunck in retrieval_chain.stream("如何评价秦始皇统一度量衡和文字对中国历史的影响？"):
    print(chunck, end="", flush=True)

C:\Users\hhm18\AppData\Local\Temp\ipykernel_24432\2266550841.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="moka-ai/m3e-base")


根据提供的文档内容，秦始皇统一度量衡和文字对中国历史产生了深远的影响，主要体现在以下几个方面：

1. **促进经济文化交流**：通过推行统一的计量单位和标准字体，极大地促进了帝国不同地区之间的经济文化交流，使各地之间的贸易往来更加便利。

2. **强化中央集权**：这些措施是秦始皇强化中央集权政策的重要组成部分，通过"书同文，车同轨"等政策，加强了中央对地方的控制和管理。

3. **便利政治管理**：统一的度量衡和文字体系使得税收征收、工程建设等国家事务更加规范化和高效化，为帝国的政治管理提供了重要支撑。

4. **推动文化融合**：淘汰六国形态各异的文字，创立小篆作为官方标准字体，有助于消除文化隔阂，促进不同地区文化的融合与发展。

总的来说，秦始皇统一度量衡和文字的措施不仅在当时具有重要的现实意义，也为后世中国的统一和发展奠定了重要基础，是中国历史上具有里程碑意义的改革举措。

这里提示的输入预期是一个包含键 "context" 和 "question" 的映射。

因此，我们需要使用我们的*检索器获取上下文*，并将用户*输入传递到 "question" 键下*。

`RunnablePassthrough` 允许我们将用户的问题传递给提示和模型。